In [66]:
import os
from bs4 import BeautifulSoup
import pandas as pd
import ipywidgets as widgets
from IPython.display import clear_output, display
import requests
import json
import re

In [22]:


def get_html_content(url, output_filename="downloaded_page.html"):
    """Fetches HTML from a URL and saves it to a local file."""

    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        print(f"Fetching content from: {url}...")
        response = requests.get(url, headers=headers, timeout=10)
        response.raise_for_status()
        soup = BeautifulSoup(response.text, "html.parser")
        return soup
    except requests.exceptions.RequestException as e:
        print(f"An error occurred while fetching the URL: {e}")
        return None

def write_html_to_file(html_content, output_filename="downloaded_page.html"):
    """Saves provided HTML content to a local file."""
    try:
        with open(output_filename, "w", encoding="utf-8") as file:
            file.write(html_content.prettify())
        absolute_path = os.path.abspath(output_filename)
    except IOError as e:
        print(f"An error occurred while writing to the file: {e}")

def write_dictionary_to_json_file(data_dict, output_filename="output_data.json"):
    """Saves a Python dictionary to a JSON file."""
    try:
        with open(output_filename, "w", encoding="utf-8") as json_file:
            json.dump(data_dict, json_file, indent=4)
        absolute_path = os.path.abspath(output_filename)
    except IOError as e:
        print(f"An error occurred while writing to the JSON file: {e}")




# ABB

This converts the html conent into a json/pyton dictionary containing all the page data. 
It is this data that can then be used to parse out specific product specific information we require.

In [53]:
sku = "1SDA068056R1"
html_content = get_html_content(f"https://new.abb.com/products/{sku}")
write_html_to_file(html_content, "downloaded_page.html")

Fetching content from: https://new.abb.com/products/1SDA068056R1...


In [ ]:
def extract_abb_page_viewmodel_from_soup(soup, export_to_json=False):
    """Parses a BeautifulSoup object to extract the embedded JavaScript 'model'

    variable as a live Python dictionary.
    """
    if not soup:
        print("Invalid Soup object provided.")
        return None

    target_script = None
    for script in soup.find_all("script"):
        if script.string and "var model =" in script.string:
            target_script = script.string
            break
    if not target_script:
        print("Could not find the 'model' variable block in the HTML.")
        return None
    match = re.search(
        r"var model\s*=\s*(\{.*?\}*?\});\s*jsLibs\.push",
        target_script,
        re.DOTALL,
    )

    if not match:
        print("Regex failed to cleanly isolate the model JSON object boundary.")
        return None
    try:
        json_string = match.group(1)
        model_dict = json.loads(json_string)
        if export_to_json:
            write_dictionary_to_json_file(model_dict, "abb_page_viewmodel_data.json")
        return model_dict
    except json.JSONDecodeError as e:
        print(f"Failed to parse the extracted string into valid JSON: {e}")
        return None

html_content = get_html_content("https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f")
extracted_model = extract_abb_page_viewmodel_from_soup(html_content, export_to_json=True)
extracted_model
write_dictionary_to_json_file(extracted_model, "abb_page_viewmodel_data.json")

Fetching content from: https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f...


In [65]:
def extract_abb_product_info_from_viewmodel(model_dict):
   """Extracts specific product information from the ABB page viewmodel dictionary."""
   product_info = model_dict.get("ProductViewModel", {}).get("Product", {})
   image_list = (product_info.get("productDetails", {}).get("item", {}).get("images", []))
   urls = [img.get("url") for img in image_list if "url" in img]
   
   attribute_groups = product_info.get("attributeGroups", {}).get("items", [])

   nested_product_data = {
      "General Information": {
            "Display Name": model_dict.get("DisplayName", ""),
            "Short Name": model_dict.get("ShortName", ""),
            "Product URL Suffix": model_dict.get("ProductURLSuffix", ""),
            "Meta Description": model_dict.get("MetaDescription", ""),
            "Global ID": model_dict.get("GlobalId", ""),
            "Images": urls if len(urls) > 1 else (urls[0] if urls else ""),
      }
    }

   for i, attribute_group in enumerate(attribute_groups):
      group_desc = attribute_group.get("description", f"Group {i}")
      nested_product_data[group_desc] = {}
      attribute_to_attribute_data = attribute_group.get("attributes", {})
      for attribute_key, attribute_data in attribute_to_attribute_data.items():
         attribute_name = attribute_data.get("attributeName", attribute_key)
         nested_values = attribute_data.get("values", [])
         extracted_texts = [v.get("text") for v in nested_values if "text" in v]
         if not extracted_texts:
               final_value = ""
         elif len(extracted_texts) == 1:
               final_value = extracted_texts[0]
         else:
               final_value = extracted_texts # Keep them as a list inside the nested dictionary for data integrity
         nested_product_data[group_desc][attribute_name] = final_value
   return nested_product_data

def export_nested_dict_to_df(nested_dict, enable_export_to_csv = False, filename="abb_attributes.csv"):
    """Flattens a nested product dictionary and exports it to a structured CSV file."""
    if not nested_dict: return None
    rows = []
    for g_name, attrs in nested_dict.items():  # Loop groups
        for a_name, a_val in attrs.items():  # Loop attributes
            val = ("\n".join(a_val) if isinstance(a_val, list) else str(a_val))  # Format lists
            rows.append({"Attribute Group": g_name, "Attribute": a_name, "Attribute Value": val})
    df = pd.DataFrame(rows)  # Create DataFrame
    if enable_export_to_csv:
        df.to_csv(filename, index=False, encoding="utf-8-sig")  # Export to CSV
    return df

html_content = get_html_content("https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f")
extracted_model = extract_abb_page_viewmodel_from_soup(html_content, export_to_json=True)
extracted_product_info = extract_abb_product_info_from_viewmodel(extracted_model)
export_nested_dict_to_df(extracted_product_info, enable_export_to_csv=True)

Fetching content from: https://new.abb.com/products/1SDA067017R1/xt2n-160-tma-80-800-3p-f-f...


,Attribute Group,Attribute,Attribute Value
0,General Information,Display Name,XT2N 160 TMA 80-800 3p F F
1,General Information,Short Name,XT2N 160 TMA 80-800 3p F F
2,General Information,Product URL Suffix,XT2N 160 TMA 80-800 3p F F
3,General Information,Meta Description,C.BREAKER TMAX XT2N 160 FIXED THREE-POLE WITH ...
4,General Information,Global ID,1SDA067017R1
...,...,...,...
82,External Classifications and Standards,ETIM 10,EC000228 - Power circuit-breaker for trafo/gen...
83,External Classifications and Standards,Object Classification Code,Q
84,External Classifications and Standards,UNSPSC,39120000
85,External Classifications and Standards,WEEE Category,5. Small Equipment (No External Dimension More...


## Summary function 

In [ ]:
def get_abb_info_from_sku(sku: str, export_csv: bool=False):
   """given a sku, return the dataframe for all the attributes for that sku"""
   html_content = get_html_content(f"https://new.abb.com/products/{sku}")
   viewmodel = extract_abb_page_viewmodel_from_soup(html_content, export_to_json=False)
   product_info = extract_abb_product_info_from_viewmodel(viewmodel)
   df = export_nested_dict_to_df(product_info, enable_export_to_csv=export_csv)
   return df

get_abb_info_from_sku("1SDA068056R1")

Fetching content from: https://new.abb.com/products/1SDA068056R1...


,Attribute Group,Attribute,Attribute Value
0,ABB EcoSolutions,ABB EcoSolutions,Yes
1,ABB EcoSolutions,ABB Site Meeting Group Waste To Landfill Target,UL 2799 Zero Waste To Landfill Validation avai...
2,ABB EcoSolutions,End Of Life Disassembling Instructions,9AKK108468A2350
3,ABB EcoSolutions,Environmental Product Declaration - EPD,9AKK108468A2294\n9AKK108468A7794
4,ABB EcoSolutions,Extended Product Lifetime,Product Modularity
...,...,...,...
75,External Classifications and Standards,ETIM 10,EC000228 - Power circuit-breaker for trafo/gen...
76,External Classifications and Standards,Object Classification Code,Q
77,External Classifications and Standards,UNSPSC,39120000
78,External Classifications and Standards,WEEE Category,5. Small Equipment (No External Dimension More...


In [68]:
def create_sku_scraper_widget():
    """Creates an interactive UI to fetch a SKU, display its DataFrame, and optionally save it."""
    # 1. Initialize UI elements
    sku_input = widgets.Text(
        value="1SDA067017R1", description="SKU:"
    )  # Text input
    save_check = widgets.Checkbox(
        value=False, description="Save CSV to PC?"
    )  # Save condition
    btn = widgets.Button(
        description="Fetch & Process", button_style="danger"
    )  # Submit button
    out = widgets.Output()  # Container to render dataframe dynamically

    # 2. Main click handling logic
    def on_button_clicked(b):
        with out:
            clear_output(wait=True)  # Refresh container area
            sku = sku_input.value.strip()
            if not sku:
                print("Please enter a valid SKU.")
                return

            # Construct the targeting address
            url = f"https://new.abb.com/products/{sku}"

            print(f"Status: Fetching data for {sku}...")
            soup = get_html_content(url)  # Fetch html soup
            if not soup:
                return

            print("Status: Parsing raw parameters...")
            model = extract_abb_model_from_soup(soup)  # Grab JS block dict
            if not model:
                return

            print("Status: Structuring nested data dictionary...")
            nested_data = extract_abb_product_info_from_viewmodel(
                model
            )  # Clean nested formatting

            # Determine whether to execute saving protocols
            filename = f"abb_{sku}_specs.csv"
            if save_check.value:
                df = export_nested_dict_to_csv(
                    nested_data, filename=filename
                )  # Flattens & saves
            else:
                # Compile dataframe in-memory only
                rows = [
                    {"Group": g, "Attribute": a, "Value": v}
                    for g, attrs in nested_data.items()
                    for a, v in attrs.items()
                ]
                df = pd.DataFrame(rows)

            clear_output()  # Wipe out tracking logs
            display(df)  # Render clean layout to notebook window

    btn.on_click(on_button_clicked)  # Bind click event helper

    # 3. Pack UI components into a structural box layout
    ui_box = widgets.VBox(
        [widgets.HBox([sku_input, save_check, btn]), out]
    )  # Horizontal control panel row
    display(ui_box)

create_sku_scraper_widget()

# Hager

## Get the product page url

In [ ]:
def query_hager_product_sitemap():
    """Fetches the global AU sitemap and builds a quick Ctrl+F lookup index."""
    url = "https://hager.com/au/products/media/sitemap_au.xml"
    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        res = requests.get(url, headers=headers, timeout=15)
        soup = BeautifulSoup(res.text, "xml")  # Parse xml tags
        urls = [loc.text for loc in soup.find_all("loc")]  # Get all links
        sku_to_url = {}
        for u in urls:
            slug = u.split("/")[-1]  # Get final string segment
            match = re.match(r"^([a-z0-9]+)-", slug)  # Snip SKU before first dash
            if match:
                sku_to_url[match.group(1).upper()] = u  # Map uppercase key
        return sku_to_url
    except Exception as e:
        print(f"Sitemap lookup map failed: {e}")
        return {}

hager_sku_to_url = query_hager_product_sitemap()
hager_sku_to_url['HEC041H']

'https://hager.com/au/products/product-information/hec041h-mccb-h250-4p-70ka-40a-lsi'

## get the hager html content

In [77]:
def get_hager_soup(sku, sku_to_url, download_html=False):
    """Looks up a SKU in the mapping dictionary and returns its BeautifulSoup page soup."""
    url = sku_to_url.get(sku.upper().strip())  # Get matching URL from dict lookup
    if not url:
        return None  # Return None early if SKU is missing

    headers = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
    try:
        res = requests.get(url, headers=headers, timeout=10)  # Fetch page html
        res.raise_for_status()  # Verify request success
        if download_html:
            with open(f"{sku}.html", "w") as f:
                f.write(res.text)
        return BeautifulSoup(res.text, "html.parser")  # Return live soup object
    except Exception as e:
        print(f"Request failed for {sku}: {e}")  # Handle request exceptions cleanly
        return None

hager_html_content = get_hager_soup("HEC041H", hager_sku_to_url, download_html=True)